In [ ]:
"""! @brief Statistic Analysis with Friedman and Wilcoxon"""
# @file StatisticTest.ipynb
#
# @mainpage Statistic Analysis
#
# @section description_main Description
# This project performs a statistical analysis for comparing multiple populations using hypothesis testing and descriptive statistics.
# The objective is to evaluate whether there are significant differences between groups based on their performance on 30 experiments,
# central tendency, and variability. The study includes multiple population comparisons to determine if observed
# differences are statistically significant under a chosen confidence level.
#
# @section libraries_main Libraries
# - numpy (https://numpy.org/)
#   - Used to perform mathematical operations
# - scipy (https://scipy.org/)
#   - Used to evaluate probability distributions
# - statsmodels (https://www.statsmodels.org/stable/index.html)
#   - Used to perform statistical tests
# - itertools (https://docs.python.org/3/library/itertools.html)
#   - Used to generate combinations of elements in a list
#
# @section authors_main Authors
# - Alejandro Cerdas
# - Kener Castillo
# - Pablo Perez

In [ ]:
# Imports
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations

## Datos

In [ ]:
# @brief Statistical results obtained from 30 independent iterations of the models.

# Results of the 30 experiments with the BERT Model
BERT = np.array([
    0.4807, 0.4799, 0.4824, 0.4774, 0.4849, 0.4766, 0.4783, 0.4770,
    0.4783, 0.4788, 0.4788, 0.4801, 0.4782, 0.4749, 0.4780, 0.4797,
    0.4824, 0.4745, 0.4789, 0.4802, 0.4810, 0.4758, 0.4824, 0.4785,
    0.4784, 0.4807, 0.4779, 0.4774, 0.4747, 0.4764,
])

# Results of the 30 experiments with the TF-IDF Model
TFIDF = np.array([
    0.4876, 0.4803, 0.4832, 0.4792, 0.4794, 0.4805, 0.4834, 0.4852,
    0.4781, 0.4859, 0.4753, 0.4791, 0.4827, 0.4795, 0.4788, 0.4812,
    0.4814, 0.4819, 0.4760, 0.4748, 0.4781, 0.4782, 0.4795, 0.4761,
    0.4761, 0.4799, 0.4796, 0.4817, 0.4816, 0.4810,
])

# Results of the 30 experiments with the SmolLM3 Model without shots
SMOLLM3_SIN_SHOTS = np.array([
    0.488905325443787, 0.5150602409638554, 0.48628048780487804,
    0.4809451219512195, 0.5129573170731707, 0.4880774962742176,
    0.5069801616458487, 0.4958615500376223, 0.50187265917603,
    0.5143072289156626, 0.5038402457757296, 0.48153730218538054,
    0.5195783132530121, 0.5123226288274833, 0.4954819277108434,
    0.4771341463414634, 0.49776119402985075, 0.5193452380952381,
    0.48134044173648133, 0.4713855421686747, 0.49255952380952384,
    0.5003717472118959, 0.5, 0.5191873589164786, 0.4915514592933948,
    0.5163249810174639, 0.4992378048780488, 0.49276466108149275,
    0.48698884758364314, 0.4784905660377359,
])

# Results of the 30 experiments with the SmolLM3 Model with 2 shots
SMOLLM3_2_SHOTS = np.array([
    0.4948224852071006, 0.49623493975903615, 0.5022865853658537,
    0.504950495049505, 0.4969512195121951, 0.5216095380029806,
    0.4996326230712711, 0.510158013544018, 0.49887640449438203,
    0.5015060240963856, 0.5084485407066052, 0.4777694046721929,
    0.4894578313253012, 0.5093353248693054, 0.49849397590361444,
    0.4634146341463415, 0.4847356664184661, 0.5133928571428571,
    0.48514851485148514, 0.4932228915662651, 0.4880952380952381,
    0.5200892857142857, 0.5038109756097561, 0.5214446952595937,
    0.4869431643625192, 0.5195488721804511, 0.4878048780487805,
    0.49047981721249045, 0.47505584512285925, 0.5049056603773585,
])

# Results of the 30 experiments with the Phi-4 Model without shots
PHI4_SIN_SHOTS = np.array([
    0.4822485207100592, 0.5075301204819277, 0.4923780487804878,
    0.5060975609756098, 0.5121951219512195, 0.503725782414307,
    0.4915378955114054, 0.5026335590669676, 0.4868913857677903,
    0.526355421686747, 0.5092165898617511, 0.4860587792012057,
    0.5060240963855421, 0.48618371919342795, 0.516566265060241,
    0.5060975609756098, 0.4873134328358209, 0.4955357142857143,
    0.5201520912547528, 0.4977409638554217, 0.47023809523809523,
    0.46651785714285715, 0.506859756097561, 0.5063957863054929,
    0.49385560675883255, 0.49097744360902257, 0.4946646341463415,
    0.5072353389185073, 0.47505584512285925, 0.4950943396226415,
])

# Results of the 30 experiments with the Phi-4 Model with 2 shots
PHI4_2_SHOTS = np.array([
    0.48298816568047337, 0.509789156626506, 0.48628048780487804,
    0.5053353658536586, 0.4961890243902439, 0.49254843517138597,
    0.49080206033848417, 0.5052473763118441, 0.4891385767790262,
    0.49623493975903615, 0.5023041474654378, 0.49736247174076864,
    0.5135542168674698, 0.5041075429424944, 0.4894578313253012,
    0.4878048780487805, 0.3777027027027027, 0.4836309523809524,
    0.5026656511805027, 0.45508100147275404, 0.49181547619047616,
    0.4880952380952381, 0.5022865853658537, 0.5041384499623778,
    0.49923195084485406, 0.49849624060150377, 0.4817073170731707,
    0.48861911987860396, 0.4802680565897245, 0.4950943396226415,
])

In [ ]:
# @brief Data structure with all the used models and their results

# Name and result of the models
treatments = {
    "TF-IDF": TFIDF,
    "BERT": BERT,
    "SmolLM3 sin shots": SMOLLM3_SIN_SHOTS,
    "SmolLM3 2 shots": SMOLLM3_2_SHOTS,
    "Phi-4 sin shots": PHI4_SIN_SHOTS,
    "Phi-4 2 shots": PHI4_2_SHOTS,
}

## Análisis estadístico

In [ ]:
def main():
    """! Execution of the statistical test.

    @Details
    Execution of the Friedman test to compare the accuracy mean of the models and Post-hoc Wilcoxon test to compare the accuracy mean of the pairs of models.

    @return None.
    """
    # List of the names of the treatments.
    names = list(treatments)

    # List of the values of the treatments.
    values = [treatments[n] for n in names]

    # Matrix where each column corresponds to a treatment.
    matrix = np.column_stack(values)

    # Friedman test to compare accuracy means of the models
    stat, p_friedman = friedmanchisquare(*values)

    # Compute the range of each treatment and the average range of the treatments.
    ranges = np.argsort(np.argsort(-matrix, axis=1), axis=1) + 1
    ranges_means = ranges.mean(axis=0)

    # Print the results
    print("Treatments Sumary")
    for name, v, r in zip(names, values, ranges_means):
        print(f"{name:20s} mean={v.mean():.6f} std={v.std(ddof=1):.6f} medium_range={r:.3f}")

    print("\nFriedman test")
    print(f"chi2 = {stat:.6f}  p = {p_friedman:.6g}")

    # Post-hoc Wilcoxon
    # List to store the comparisons
    comparisons = []

    # To save the p values without correction
    p_vals = []

    # Iterate over the combinations of names
    for i, j in combinations(range(len(names)), 2):

        # Test of Wilcoxon paired test between two treatments.
        w, p = wilcoxon(values[i], values[j], zero_method="wilcox")

        comparisons.append((names[i], names[j], w, p))
        p_vals.append(p)

    # Holm correction to control multiple comparisons.
    _, p_holm, _, _ = multipletests(p_vals, method='holm')

    # Print the results
    print("\nPost-hoc Wilcoxon paired")

    # Iterate over pairwise comparisons and their Holm-adjusted p-values
    for (a, b, w, p), ph in zip(comparisons, p_holm):
        sig = "Yes" if ph < 0.05 else "No"
        # Print the information about the test
        print(f"{a:20s} vs {b:20s} W={w:7.3f} p={p:.6g} p_holm={ph:.6g} sig={sig}")

# Execute the statistical test
main()